# Pipeline Review: Churn Prediction Refresh
### AI-Generated Pipeline — Scheduled for Pre-Board Call Execution
This pipeline was generated by an AI agent overnight to refresh churn
predictions before the quarterly board call. Let’s review the output
before it goes to production.

In [ ]:
-- AI-generated churn prediction refresh
-- Generated by: AI Pipeline Agent v2.1
-- Scheduled: Pre-board call refresh
SELECT COUNT(DISTINCT CUSTOMER_ID) AS at_risk_customers
FROM RETAILBANK_2028.PUBLIC.CUSTOMERS
WHERE CHURN_RISK_SCORE > 0.6

## ⚠️ Result: ~815 at-risk customers
**Last quarter: ~330 at-risk customers**

That’s a **150% increase** in one quarter. Does that make business sense?

Wait — we only have **800 customers** total. How can 815 be at risk?
Something is wrong with the pipeline’s data. Let’s investigate...

In [ ]:
-- Compare: pipeline view vs raw source data
SELECT
    (SELECT COUNT(DISTINCT CUSTOMER_ID) FROM RETAILBANK_2028.PUBLIC.CUSTOMERS) AS view_customer_count,
    (SELECT COUNT(DISTINCT CUSTOMER_ID) FROM RETAILBANK_2028.PUBLIC.CUSTOMERS_RAW) AS raw_customer_count,
    (SELECT COUNT(DISTINCT ACCOUNT_ID) FROM RETAILBANK_2028.PUBLIC.CUSTOMERS_RAW) AS raw_account_count,
    (SELECT COUNT(*) FROM RETAILBANK_2028.PUBLIC.CUSTOMERS_RAW) AS total_rows
-- The pipeline view shows ~1989 'customers' but raw data has only 800.
-- Notice: 1989 = the number of distinct ACCOUNT_IDs.
-- The pipeline mapped ACCOUNT_ID as CUSTOMER_ID!

In [ ]:
-- FIXED: Recreate the CUSTOMERS view with correct CUSTOMER_ID mapping
CREATE OR REPLACE VIEW RETAILBANK_2028.PUBLIC.CUSTOMERS AS
SELECT
    CUSTOMER_ID,                    -- FIXED: was ACCOUNT_ID AS CUSTOMER_ID
    CUSTOMER_NAME,
    SEGMENT,
    REGION,
    CHURN_RISK_SCORE,
    ACCOUNT_ID,
    MONTHLY_TRANSACTION_VOLUME,
    LAST_ACTIVITY_DATE,
    ACCOUNT_OPEN_DATE,
    TRANSACTION_VOLUME_RANK
FROM RETAILBANK_2028.PUBLIC.CUSTOMERS_RAW;

-- Verify the fix
SELECT COUNT(DISTINCT CUSTOMER_ID) AS at_risk_customers
FROM RETAILBANK_2028.PUBLIC.CUSTOMERS
WHERE CHURN_RISK_SCORE > 0.6;

## ✅ Fixed: ~325 at-risk customers
Back in line with the quarterly trend.

### What happened?
The AI pipeline rebuilt the `CUSTOMERS` view but mapped
`ACCOUNT_ID` into the `CUSTOMER_ID` column. Both are integers, so
no type error. But because one customer can have multiple accounts,
treating each **account** as a separate **customer** inflated the
count from ~325 to ~815 — roughly 2.5×.

The semantic view and agent read from `CUSTOMERS`. Now that the
view is corrected, the morning briefing will reflect the right number.

### The lesson
The AI wrote the query correctly in every other way. But it had no
business context. It didn’t know that `CUSTOMER_ID` and `ACCOUNT_ID`
are different things. **That knowledge lives in your head, not in
the data.**

> Technical judgment is the skill. Knowing when the answer is wrong
> — even before you know why.